In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pybedtools import BedTool, Interval

import subprocess

import scanpy as sc
import anndata

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.stats import zscore
import pyranges as pr

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.columns = ['chrom', 'start', 'end']
pmds.shape

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.header.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t')
peaks.rename(columns={'seqnames' : 'chrom'}, inplace=True)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks['chrom'], peaks['start'])]
peaks.shape

### rna expression lower on average in PMDs?

In [ ]:
adata_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/h5ad/atac/peaks/clusters/justin_chromhmm_cluster_donor.h5ad'

adata = sc.read_h5ad(adata_path)
adata.var.index = [f'{x}_{y}' for x,y in zip(adata.var['seqnames'], adata.var['start'])]

In [ ]:
adata

In [ ]:
pseudo_df = pd.DataFrame(adata.X, index=adata.obs['cluster_donor'], columns=adata.var.index)
pseudo_df.shape

In [ ]:
# ### subset adata to autosome
# auto_adata

In [ ]:
denom = pseudo_df.sum(axis=1)
norm_pseudo = pseudo_df.div(denom, axis=0) * 1e6
norm_pseudo.columns = adata.var.index
norm_pseudo.shape

In [ ]:
log_pseudo = np.log(norm_pseudo + 1)
log_pseudo.shape

### split all genes into PMDs/non-PMDS

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
peaks_bt = BedTool.from_dataframe(peaks)
boundaries = []
for p in pmds_bt:
    boundaries.append(Interval(p.chrom, p.start, p.start+1))
    boundaries.append(Interval(p.chrom, p.end-1, p.end))

boundaries = BedTool(boundaries)
res = peaks_bt.closest(boundaries, d=True)
peaks_dist = res.to_dataframe(header=None)
peaks_dist.index = [f'{x}_{y}' for x,y in zip(peaks_dist[0], peaks_dist[1])]
peaks_dist = peaks_dist[~peaks_dist.index.duplicated()]
peaks_dist.shape

In [ ]:
peaks_dist.head()

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
peaks_bt = BedTool.from_dataframe(peaks)
peaks_pmds = peaks_bt.intersect(pmds_bt, wa=True).to_dataframe(header=None)
peaks_pmds.index = [f'{x}_{y}' for x,y in zip(peaks_pmds[0], peaks_pmds[1])]
peaks_pmds.shape

In [ ]:
peaks_pmds.head()

In [ ]:
s = 'Start'
mask = [x.split('_')[0] == s for x in norm_pseudo.index]

tmp_pseudo = norm_pseudo[mask]
tmp_pseudo.shape

In [ ]:
merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
merged_expr.shape

In [ ]:
mask = merged_expr.index.isin(peaks_pmds.index)
merged_pmds = merged_expr[mask].copy()
merged_pmds.shape

In [ ]:
mask = ~merged_expr.index.isin(peaks_pmds.index)
merged_comp = merged_expr[mask].copy()
merged_comp.shape

In [ ]:
peaks_dist

In [ ]:
merged_pmds['type'] = 'pmds'
merged_comp['type'] = 'non-pmds'
to_plot = pd.concat([merged_pmds, merged_comp])
to_plot.columns = ['expr', 'type']
mask = to_plot['expr'] != 0
to_plot = to_plot[mask]

###
to_plot['dist'] = peaks_dist.reindex(to_plot.index)[21]
mask = to_plot['type'] == 'pmds'
to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']

to_plot.shape

In [ ]:
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'type'

ax = sns.histplot(data=to_plot, x=x_lab, y=y_lab)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - all gene expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'type'

ax = sns.scatterplot(data=to_plot, x=x_lab, y=y_lab, size=1)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - all peaks inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
to_plot.to_csv(f'{outdir}/figure2/atac_peaks_dist_start.csv', sep='\t', index=None)

### try DEGs now

In [ ]:
s = 'Start'
tmp_peaks = peaks[peaks['coarse_v7'] == s]
tmp_peaks = tmp_peaks[~tmp_peaks['chrom'].isna()]
tmp_peaks['start'] = tmp_peaks['start'].astype(int)
tmp_peaks['end'] = tmp_peaks['end'].astype(int)
tmp_peaks.shape

In [ ]:
tmp_peaks.head()

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
peaks_bt = BedTool.from_dataframe(tmp_peaks)
peaks_pmds = peaks_bt.intersect(pmds_bt, wa=True).to_dataframe(header=None)
peaks_pmds.index = [f'{x}_{y}' for x,y in zip(peaks_pmds[0], peaks_pmds[1])]

peaks_comp = peaks_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe(header=None)
peaks_comp.index = [f'{x}_{y}' for x,y in zip(peaks_comp[0], peaks_comp[1])]

mask = [x.split('_')[0] == s for x in norm_pseudo.index]
tmp_pseudo = norm_pseudo[mask]
tmp_pseudo.shape

merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
mask = merged_expr.index.isin(peaks_pmds.index)
merged_pmds = merged_expr[mask].copy()

mask = merged_expr.index.isin(peaks_comp.index)
merged_comp = merged_expr[mask].copy()
merged_pmds.shape, merged_comp.shape

In [ ]:
merged_pmds['type'] = 'pmds'
merged_comp['type'] = 'non-pmds'
to_plot = pd.concat([merged_pmds, merged_comp])
to_plot.columns = ['expr', 'type']
mask = to_plot['expr'] != 0
to_plot = to_plot[mask]

###
to_plot['dist'] = peaks_dist.reindex(to_plot.index)[21]
mask = to_plot['type'] == 'pmds'
to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']

to_plot.shape

In [ ]:
tmp = to_plot[to_plot['type'] == 'pmds']
tmp['dist'], tmp['dist'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'type'

ax = sns.scatterplot(data=to_plot, x=x_lab, y=y_lab, size=1)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - differential peaks inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

### do all peaks for all clusters

In [ ]:
clust_list = []
for s in tmp_order:
    ###
    tmp_peaks = peaks.copy()
    ###
    pmds_bt = BedTool.from_dataframe(pmds)
    peaks_bt = BedTool.from_dataframe(tmp_peaks)
    peaks_pmds = peaks_bt.intersect(pmds_bt, wa=True).to_dataframe(header=None)
    peaks_pmds.index = [f'{x}_{y}' for x,y in zip(peaks_pmds[0], peaks_pmds[1])]

    peaks_comp = peaks_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe(header=None)
    peaks_comp.index = [f'{x}_{y}' for x,y in zip(peaks_comp[0], peaks_comp[1])]

    mask = [x.split('_')[0] == s for x in norm_pseudo.index]
    tmp_pseudo = norm_pseudo[mask]
    tmp_pseudo.shape

    merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
    mask = merged_expr.index.isin(peaks_pmds.index)
    merged_pmds = merged_expr[mask].copy()

    mask = merged_expr.index.isin(peaks_comp.index)
    merged_comp = merged_expr[mask].copy()
    merged_pmds.shape, merged_comp.shape
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    to_plot = pd.concat([merged_pmds, merged_comp])
    to_plot.columns = ['expr', 'type']
    mask = to_plot['expr'] != 0
    to_plot = to_plot[mask]

    ###
    to_plot['dist'] = peaks_dist.reindex(to_plot.index)[21]
    mask = to_plot['type'] == 'pmds'
    to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']

    ###
    to_plot['dist'] = peaks_dist.reindex(to_plot.index)[21]
    mask = to_plot['type'] == 'pmds'
    to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']

    to_plot['cluster'] = s
    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'cluster'

ax = sns.scatterplot(data=pmds_plot, x=x_lab, y=y_lab, hue=hue, size=1)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - all gene expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
pmds_plot['dist'].head()

In [ ]:
pos = pmds_plot[pmds_plot['dist'] > 0]
neg = pmds_plot[pmds_plot['dist'] < 0]
pos.shape, neg.shape

In [ ]:
pos['expr'].mean(), neg['expr'].mean()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
pmds_plot.to_csv(f'{outdir}/figure2/atac_all_peaks_dist.csv', sep='\t', index=None)

### do differential peaks for all clusters

In [ ]:
clust_list = []
for s in tmp_order:
    ###
    tmp_peaks = peaks[peaks['coarse_v7'] == s]
    tmp_peaks = tmp_peaks[~tmp_peaks['chrom'].isna()]
    tmp_peaks['start'] = tmp_peaks['start'].astype(int)
    tmp_peaks['end'] = tmp_peaks['end'].astype(int)

    ###
    pmds_bt = BedTool.from_dataframe(pmds)
    peaks_bt = BedTool.from_dataframe(tmp_peaks)
    peaks_pmds = peaks_bt.intersect(pmds_bt, wa=True).to_dataframe(header=None)
    peaks_pmds.index = [f'{x}_{y}' for x,y in zip(peaks_pmds[0], peaks_pmds[1])]

    peaks_comp = peaks_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe(header=None)
    peaks_comp.index = [f'{x}_{y}' for x,y in zip(peaks_comp[0], peaks_comp[1])]

    ###
    mask = [x.split('_')[0] == s for x in norm_pseudo.index]
    tmp_pseudo = norm_pseudo[mask]

    merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
    mask = merged_expr.index.isin(peaks_pmds.index)
    merged_pmds = merged_expr[mask].copy()

    mask = merged_expr.index.isin(peaks_comp.index)
    merged_comp = merged_expr[mask].copy()
    
    ###
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    to_plot = pd.concat([merged_pmds, merged_comp])
    to_plot.columns = ['expr', 'type']
    mask = to_plot['expr'] != 0
    to_plot = to_plot[mask]

    ###
    to_plot['dist'] = peaks_dist.reindex(to_plot.index)[21]
    mask = to_plot['type'] == 'pmds'
    to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']
    to_plot['cluster'] = s
    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'cluster'

ax = sns.scatterplot(data=pmds_plot, x=x_lab, y=y_lab, hue=hue, size=1)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('ATAC - differential peaks inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
pos = pmds_plot[pmds_plot['dist'] > 0]
neg = pmds_plot[pmds_plot['dist'] < 0]
pos.shape, neg.shape

In [ ]:
pos['expr'].mean(), neg['expr'].mean()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
pmds_plot.to_csv(f'{outdir}/figure2/atac_diff_peaks_dist.csv', sep='\t', index=None)

In [ ]:
!date